# 4050. Minimum Days to Score Exactly N Points — Nhập môn Quy hoạch động (Dynamic Programming) kiểu "đổi tiền xu"

Bài toán LeetCode (Medium):

```python
class Solution:
    def minDays(self, n: int) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- **Số tam giác** `1, 3, 6, 10, ...` và công thức `L(L+1)/2`.
- Cách **mô hình hoá** lịch "chuỗi ngày + ngày nghỉ" thành bài toán **chọn các số tam giác có tổng bằng n**.
- Mẹo "**tính trước ngày nghỉ vào mỗi chuỗi rồi trừ 1**".
- Lời giải **đệ quy vét cạn**, vì sao nó bùng nổ.
- **Memoization** (`functools.lru_cache`) — quy hoạch động **từ trên xuống**.
- **DP từ dưới lên** (bottom-up) theo kiểu bài **Coin Change** — lời giải chính `O(n√n)`.
- **Truy vết** để in ra lịch cụ thể, và kỹ thuật **tính trước một lần** cho mọi test.
- Vì sao **tham lam** sai, vì sao "**tối đa 3 chuỗi**" (định lý Gauss) cũng sai.
- Lỗi hay mắc, biến thể, bài tập có đáp án.

# 1. Đề bài nói gì?

Bạn có điểm ban đầu là `0`. Mỗi ngày bạn **kiếm điểm** hoặc **nghỉ**.

- Điểm được kiếm theo **chuỗi** (streak): ngày thứ nhất của chuỗi được `1` điểm, ngày thứ hai `2` điểm, ngày thứ ba `3` điểm, ...
- **Nghỉ** một ngày: không được điểm và **chuỗi bị reset** → lần kiếm điểm tiếp theo lại bắt đầu từ `1`.

Trả về số ngày **ít nhất** (tính cả ngày nghỉ) để đạt **đúng** `n` điểm.

### Ví dụ

| n | Kết quả | Lịch (K = kiếm, N = nghỉ) |
|---|---:|---|
| 2 | 3 | `K(1) N K(1)` → 1 + 1 = 2 |
| 9 | 6 | `K(1) K(2) K(3) N K(1) K(2)` → 6 + 3 = 9 |
| 12 | 7 | `K(1) K(2) K(3) N K(1) K(2) K(3)` → 6 + 6 = 12 |

Với `n = 2`: nếu kiếm 2 ngày liền sẽ được `1 + 2 = 3` điểm — **vượt** quá. Đề yêu cầu **đúng** `n`, nên phải nghỉ một ngày.

### Ràng buộc

- `1 <= n <= 10^5`

# 2. Nền tảng — Số tam giác

Một chuỗi dài `L` ngày cho tổng điểm:

```text
1 + 2 + 3 + ... + L = L * (L + 1) / 2
```

Các số này gọi là **số tam giác** (vì xếp được thành hình tam giác chấm bi):

| L | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| T(L) | 1 | 3 | 6 | 10 | 15 | 21 | 28 |

Trong Python dùng `//` (chia lấy phần nguyên) để kết quả là số nguyên: `L * (L + 1) // 2`. Tích `L*(L+1)` luôn chẵn nên phép chia luôn chính xác.

In [1]:
def triangular(L):
    return L * (L + 1) // 2

print([triangular(L) for L in range(1, 11)])
print("Kiểm tra bằng sum:", sum(range(1, 8)), "==", triangular(7))

[1, 3, 6, 10, 15, 21, 28, 36, 45, 55]
Kiểm tra bằng sum: 28 == 28


# 3. Mô hình hoá lịch trình

## Quan sát 1 — Không bao giờ nghỉ thừa

- Nghỉ ở **đầu** hoặc **cuối** lịch: vô ích, bỏ đi sẽ ít ngày hơn.
- Nghỉ **hai ngày liền**: ngày nghỉ thứ hai vô ích (chuỗi đã reset rồi).

Vậy lịch tối ưu có dạng:

```text
[chuỗi L1] N [chuỗi L2] N [chuỗi L3] ... N [chuỗi Lk]
```

## Quan sát 2 — Đếm ngày và điểm

```text
điểm  = T(L1) + T(L2) + ... + T(Lk)              (phải đúng bằng n)
số ngày = L1 + L2 + ... + Lk + (k - 1)           (k - 1 ngày nghỉ ở giữa)
```

Thứ tự các chuỗi **không quan trọng** — chỉ cần chọn **một tập (có thể lặp) các số tam giác** có tổng bằng `n`.

## Mẹo "gộp ngày nghỉ vào chuỗi"

`k - 1` ngày nghỉ khó xử lý vì phụ thuộc số chuỗi. Mẹo: **tạm tính** mỗi chuỗi tốn `L + 1` ngày (chính nó + một ngày nghỉ phía sau). Khi đó:

```text
Σ (Li + 1) = Σ Li + k = (số ngày thật) + 1
```

Vì chuỗi **cuối cùng** không cần ngày nghỉ phía sau, ta chỉ việc **trừ 1** ở cuối.

> **Bài toán mới:** chọn các số tam giác (được lặp lại) có tổng **đúng `n`**, mỗi số `T(L)` có **giá** `L + 1`. Tìm **tổng giá nhỏ nhất**, rồi trừ 1.

Đây chính là bài **Coin Change** (đổi tiền xu): "đồng xu" mệnh giá `T(L)`, mỗi đồng tốn `L + 1`.

In [2]:
# Kiểm tra mẹo trên ví dụ n = 9: hai chuỗi L = 3 và L = 2
streaks = [3, 2]
points = sum(triangular(L) for L in streaks)
real_days = sum(streaks) + (len(streaks) - 1)
trick_days = sum(L + 1 for L in streaks) - 1
print("điểm =", points, "| ngày thật =", real_days, "| theo mẹo =", trick_days)

điểm = 9 | ngày thật = 6 | theo mẹo = 6


# 4. Cách 1 — Đệ quy vét cạn

Gọi `f(rem)` = tổng giá nhỏ nhất để gom đủ **đúng** `rem` điểm (theo mẹo "L + 1").

- `f(0) = 0` (không cần chuỗi nào).
- Với `rem > 0`: thử **chuỗi đầu tiên** dài `L` với mọi `T(L) <= rem`:

```text
f(rem) = min over L of [ (L + 1) + f(rem - T(L)) ]
```

Đáp án: `f(n) - 1`.

Luôn có cách giải vì `T(1) = 1` (có thể dùng toàn chuỗi dài 1).

In [3]:
call_count = 0

def f_bruteforce(rem):
    global call_count
    call_count += 1
    if rem == 0:
        return 0
    best = float("inf")
    L = 1
    while triangular(L) <= rem:
        best = min(best, (L + 1) + f_bruteforce(rem - triangular(L)))
        L += 1
    return best


def min_days_bruteforce(n):
    return f_bruteforce(n) - 1


for n in [2, 9, 12]:
    call_count = 0
    print(f"n = {n}: {min_days_bruteforce(n)} ngày  (số lần gọi hàm: {call_count})")

n = 2: 3 ngày  (số lần gọi hàm: 3)
n = 9: 6 ngày  (số lần gọi hàm: 71)
n = 12: 7 ngày  (số lần gọi hàm: 266)


In [4]:
for n in [10, 20, 25, 30]:
    call_count = 0
    min_days_bruteforce(n)
    print(f"n = {n:2}: số lần gọi hàm = {call_count:,}")

n = 10: số lần gọi hàm = 111
n = 20: số lần gọi hàm = 8,869
n = 25: số lần gọi hàm = 79,308
n = 30: số lần gọi hàm = 709,207


Từ khoá `global call_count` cho phép hàm **sửa** biến `call_count` nằm ngoài hàm (mặc định Python coi biến được gán trong hàm là biến cục bộ).

### Vì sao bùng nổ?

Số lần gọi tăng **theo cấp số nhân**. Nguyên nhân: cùng một `f(rem)` bị tính **lặp đi lặp lại** qua nhiều đường khác nhau. Ví dụ `f(9)` gọi `f(8)` (bỏ chuỗi 1) và `f(6)` (bỏ chuỗi 3); `f(8)` lại gọi `f(7)`, `f(5)`, `f(2)`; `f(7)` gọi `f(6)` lần nữa...

Đây là dấu hiệu kinh điển của **bài toán con chồng lấp** (overlapping subproblems) → dùng **quy hoạch động**.

# 5. Cách 2 — Memoization (DP từ trên xuống)

**Memoization** = "ghi nhớ": lần đầu tính `f(rem)` thì **lưu** lại; lần sau gặp lại chỉ việc **tra**.

Python có sẵn decorator `@lru_cache(maxsize=None)` (trong `functools`): đặt ngay trên hàm, Python tự lưu kết quả theo tham số.

In [5]:
from functools import lru_cache

@lru_cache(maxsize=None)
def slow_square(x):
    print(f"  (đang tính {x}^2...)")
    return x * x

print(slow_square(4))
print(slow_square(4))     # lần 2: không in "(đang tính...)" vì lấy từ cache

  (đang tính 4^2...)
16
16


In [6]:
import sys

def min_days_memo(n):
    @lru_cache(maxsize=None)
    def f(rem):
        if rem == 0:
            return 0
        best = float("inf")
        L = 1
        while triangular(L) <= rem:
            best = min(best, (L + 1) + f(rem - triangular(L)))
            L += 1
        return best
    return f(n) - 1


for n in [2, 9, 12, 40, 500]:
    print(f"n = {n}: {min_days_memo(n)}")

n = 2: 3
n = 9: 6
n = 12: 7
n = 40: 13
n = 500: 36


### Hạn chế của đệ quy với `n = 10^5`

Mỗi lần gọi `f(rem)` có thể gọi `f(rem - 1)` → **độ sâu đệ quy** tới `n`. Python mặc định giới hạn độ sâu khoảng `1000` (`sys.getrecursionlimit()`), vượt quá sẽ báo `RecursionError`.

Có thể nâng giới hạn bằng `sys.setrecursionlimit(...)`, nhưng đệ quy sâu vẫn chậm và tốn bộ nhớ stack. Cách sạch hơn: **đảo chiều** — tính từ nhỏ lên lớn.

In [7]:
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    min_days_memo(5000)
except RecursionError as e:
    print("RecursionError:", str(e)[:60], "...")

Giới hạn đệ quy mặc định: 1000
RecursionError: maximum recursion depth exceeded ...


`try: ... except RecursionError:` dùng để **bắt lỗi**: nếu đoạn trong `try` gây lỗi `RecursionError`, chương trình không dừng mà nhảy vào nhánh `except`.

# 6. Cách 3 — DP từ dưới lên (bottom-up)

Thay vì hỏi "để có `rem` điểm cần gì?" rồi đệ quy xuống, ta **điền bảng** `dp[s]` cho `s = 0, 1, 2, ..., n`:

```text
dp[0] = 0
dp[s] = min over L với T(L) <= s của  dp[s - T(L)] + (L + 1)
đáp án = dp[n] - 1
```

Khi tính `dp[s]`, mọi `dp[s - T(L)]` (nhỏ hơn `s`) **đã được tính xong** — không cần đệ quy.

Danh sách "đồng xu" `(T(L), L + 1)` chỉ gồm các `L` với `T(L) <= n`. Vì `T(L) ≈ L²/2`, có khoảng `√(2n)` đồng xu — với `n = 10^5` chỉ có **446** đồng xu.

In [8]:
def min_days_dp(n):
    coins = []                        # (điểm T(L), giá L + 1)
    L = 1
    while triangular(L) <= n:
        coins.append((triangular(L), L + 1))
        L += 1

    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        for points, cost in coins:
            if points > s:
                break                 # coins tăng dần -> các đồng sau còn lớn hơn
            candidate = dp[s - points] + cost
            if candidate < dp[s]:
                dp[s] = candidate
    return dp[n] - 1


print(min_days_dp(2), min_days_dp(9), min_days_dp(12))
print("Số 'đồng xu' khi n = 10^5:", sum(1 for L in range(1, 500) if triangular(L) <= 10**5))

3 6 7
Số 'đồng xu' khi n = 10^5: 446


# 7. Mô phỏng bảng DP

Đồng xu với `n = 12`: `(1, 2)`, `(3, 3)`, `(6, 4)`, `(10, 5)` — dạng `(điểm, giá)`.

| s | dùng (1, giá 2) | dùng (3, giá 3) | dùng (6, giá 4) | dùng (10, giá 5) | dp[s] |
|---|---|---|---|---|---:|
| 0 | | | | | 0 |
| 1 | dp[0]+2 = 2 | | | | 2 |
| 2 | dp[1]+2 = 4 | | | | 4 |
| 3 | dp[2]+2 = 6 | dp[0]+3 = **3** | | | 3 |
| 4 | dp[3]+2 = 5 | dp[1]+3 = 5 | | | 5 |
| 5 | dp[4]+2 = 7 | dp[2]+3 = 7 | | | 7 |
| 6 | dp[5]+2 = 9 | dp[3]+3 = 6 | dp[0]+4 = **4** | | 4 |
| 7 | dp[6]+2 = **6** | dp[4]+3 = 8 | dp[1]+4 = 6 | | 6 |
| 8 | dp[7]+2 = 8 | dp[5]+3 = 10 | dp[2]+4 = 8 | | 8 |
| 9 | dp[8]+2 = 10 | dp[6]+3 = **7** | dp[3]+4 = 7 | | 7 |
| 10 | dp[9]+2 = 9 | dp[7]+3 = 9 | dp[4]+4 = 9 | dp[0]+5 = **5** | 5 |
| 11 | dp[10]+2 = **7** | dp[8]+3 = 11 | dp[5]+4 = 11 | dp[1]+5 = 7 | 7 |
| 12 | dp[11]+2 = 9 | dp[9]+3 = 10 | dp[6]+4 = **8** | dp[2]+5 = 9 | 8 |

- `n = 2`: `dp[2] - 1 = 3` ✔
- `n = 9`: `dp[9] - 1 = 6` ✔
- `n = 12`: `dp[12] - 1 = 7` ✔ (hai chuỗi dài 3: `4 + 4 = 8`, trừ 1)

In [9]:
def trace_dp(n):
    coins = []
    L = 1
    while triangular(L) <= n:
        coins.append((triangular(L), L + 1))
        L += 1
    print("coins (điểm, giá):", coins)
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        options = []
        for points, cost in coins:
            if points > s:
                break
            options.append(f"dp[{s - points}]+{cost}={dp[s - points] + cost}")
            dp[s] = min(dp[s], dp[s - points] + cost)
        print(f"s={s:2}: " + ", ".join(options) + f"  -> dp[{s}] = {dp[s]}")
    print("Đáp án = dp[n] - 1 =", dp[n] - 1)
    return dp[n] - 1

trace_dp(12)

coins (điểm, giá): [(1, 2), (3, 3), (6, 4), (10, 5)]
s= 1: dp[0]+2=2  -> dp[1] = 2
s= 2: dp[1]+2=4  -> dp[2] = 4
s= 3: dp[2]+2=6, dp[0]+3=3  -> dp[3] = 3
s= 4: dp[3]+2=5, dp[1]+3=5  -> dp[4] = 5
s= 5: dp[4]+2=7, dp[2]+3=7  -> dp[5] = 7
s= 6: dp[5]+2=9, dp[3]+3=6, dp[0]+4=4  -> dp[6] = 4
s= 7: dp[6]+2=6, dp[4]+3=8, dp[1]+4=6  -> dp[7] = 6
s= 8: dp[7]+2=8, dp[5]+3=10, dp[2]+4=8  -> dp[8] = 8
s= 9: dp[8]+2=10, dp[6]+3=7, dp[3]+4=7  -> dp[9] = 7
s=10: dp[9]+2=9, dp[7]+3=9, dp[4]+4=9, dp[0]+5=5  -> dp[10] = 5
s=11: dp[10]+2=7, dp[8]+3=11, dp[5]+4=11, dp[1]+5=7  -> dp[11] = 7
s=12: dp[11]+2=9, dp[9]+3=10, dp[6]+4=8, dp[2]+5=9  -> dp[12] = 8
Đáp án = dp[n] - 1 = 7


7

`", ".join(list_of_strings)` nối các chuỗi trong list bằng dấu `", "` — tiện để in một dòng gọn gàng.

# 8. Truy vết: in ra lịch cụ thể

Bảng `dp` cho biết **số ngày**, nhưng không cho biết **lịch**. Để truy vết, lưu thêm `choice[s]` = độ dài chuỗi `L` đã cho ra `dp[s]` tốt nhất. Sau đó đi ngược từ `s = n`: lấy chuỗi `choice[s]`, trừ `T(L)` khỏi `s`, lặp lại đến `0`.

In [10]:
def min_days_with_schedule(n):
    coins = []
    L = 1
    while triangular(L) <= n:
        coins.append(L)
        L += 1
    INF = float("inf")
    dp = [INF] * (n + 1)
    choice = [0] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        for L in coins:
            t = triangular(L)
            if t > s:
                break
            if dp[s - t] + L + 1 < dp[s]:
                dp[s] = dp[s - t] + L + 1
                choice[s] = L
    # truy vết
    streaks = []
    s = n
    while s > 0:
        streaks.append(choice[s])
        s -= triangular(choice[s])
    # dựng lịch dạng chuỗi ký tự
    days = []
    for idx, L in enumerate(streaks):
        if idx > 0:
            days.append("N")
        days.extend(str(k) for k in range(1, L + 1))
    return dp[n] - 1, streaks, " ".join(days)


for n in [2, 9, 12, 53]:
    d, streaks, schedule = min_days_with_schedule(n)
    print(f"n = {n:3}: {d} ngày | các chuỗi {streaks} | lịch: {schedule}")

n =   2: 3 ngày | các chuỗi [1, 1] | lịch: 1 N 1
n =   9: 6 ngày | các chuỗi [2, 3] | lịch: 1 2 N 1 2 3
n =  12: 7 ngày | các chuỗi [3, 3] | lịch: 1 2 3 N 1 2 3
n =  53: 17 ngày | các chuỗi [1, 1, 3, 9] | lịch: 1 N 1 N 1 2 3 N 1 2 3 4 5 6 7 8 9


Trong lịch in ra, số `k` nghĩa là "ngày thứ `k` của chuỗi, được `k` điểm", còn `N` là ngày nghỉ.

# 9. Cách 4 — Tính trước một lần cho mọi `n`

LeetCode gọi `minDays` **nhiều lần** với các `n` khác nhau. Mỗi lần tính lại bảng `dp` tới `10^5` sẽ tốn thời gian.

Nhận xét: bảng `dp` **không phụ thuộc `n`** — `dp[s]` luôn là đáp án (cộng 1) cho mọi `s`. Vậy ta tính **một lần** bảng `dp` tới `10^5` ở **mức module**, rồi mỗi lần gọi chỉ việc trả về `dp[n] - 1` trong `O(1)`.

Bên dưới là bản nộp LeetCode theo hướng này. Để vòng lặp trong nhanh hơn, ta tính sẵn hai list `points` và `costs` và dùng biến cục bộ.

In [11]:
import time

def _build_min_days_table(limit=10**5):
    points, costs = [], []
    L = 1
    while L * (L + 1) // 2 <= limit:
        points.append(L * (L + 1) // 2)
        costs.append(L + 1)
        L += 1
    m = len(points)
    dp = [0] * (limit + 1)
    for s in range(1, limit + 1):
        best = 1 << 60                      # một số rất lớn
        for idx in range(m):
            p = points[idx]
            if p > s:
                break
            v = dp[s - p] + costs[idx]
            if v < best:
                best = v
        dp[s] = best
    return dp

t0 = time.time()
_DP = _build_min_days_table()               # chạy MỘT lần khi nạp file
print(f"Tính bảng dp tới 10^5 mất {time.time() - t0:.2f} giây")


class Solution:
    def minDays(self, n: int) -> int:
        return _DP[n] - 1

Tính bảng dp tới 10^5 mất 1.69 giây


`1 << 60` là `2^60` (dịch bit 1 sang trái 60 vị trí) — một số nguyên rất lớn, dùng thay cho "vô cùng". So sánh số nguyên nhanh hơn so sánh với `float("inf")` một chút.

In [12]:
sol = Solution()

tests = [(1, 1), (2, 3), (3, 2), (6, 3), (9, 6), (12, 7), (10, 4), (4, 4)]
for n, expected in tests:
    result = sol.minDays(n)
    print(f"n = {n:3} -> {result:3} | expected {expected:3} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

for n in range(1, 31):
    assert sol.minDays(n) == min_days_bruteforce(n) == min_days_memo(n) == min_days_dp(n)
print("4 cách khớp nhau với n = 1..30 ✔")

for n in [1000, 54321, 99999, 100000]:
    assert sol.minDays(n) == min_days_dp(n)
print("Solution khớp DP trực tiếp với n lớn ✔", sol.minDays(10**5))

n =   1 ->   1 | expected   1 | PASS
n =   2 ->   3 | expected   3 | PASS
n =   3 ->   2 | expected   2 | PASS
n =   6 ->   3 | expected   3 | PASS
n =   9 ->   6 | expected   6 | PASS
n =  12 ->   7 | expected   7 | PASS
n =  10 ->   4 | expected   4 | PASS
n =   4 ->   4 | expected   4 | PASS


4 cách khớp nhau với n = 1..30 ✔


Solution khớp DP trực tiếp với n lớn ✔ 481


Giải thích vài test tự thêm:

- `n = 1`: chuỗi dài 1 → 1 ngày.
- `n = 3`: chuỗi dài 2 (`1 + 2`) → 2 ngày.
- `n = 4`: `3 + 1` → `K K N K` = 4 ngày (hoặc `1 + 1 + 1 + 1` cần 7 ngày).
- `n = 10`: chuỗi dài 4 → 4 ngày.

# 10. Vì sao tham lam sai? Vì sao "tối đa 3 chuỗi" sai?

## Tham lam (greedy)

Ý nghĩ tự nhiên: "luôn chọn chuỗi **dài nhất** có thể (số tam giác lớn nhất `<= phần còn lại`)". Chuỗi dài hiệu quả hơn (điểm/ngày cao hơn), nên nghe hợp lý.

Ta để máy tìm phản ví dụ.

In [13]:
def min_days_greedy(n):
    days, rem, k = 0, n, 0
    while rem > 0:
        L = 1
        while triangular(L + 1) <= rem:
            L += 1
        rem -= triangular(L)
        days += L
        k += 1
    return days + (k - 1)

counter = [n for n in range(1, 200) if min_days_greedy(n) != sol.minDays(n)]
print("Các n mà tham lam sai (n < 200):", counter[:10], "...")
n = counter[0]
print(f"n = {n}: tham lam = {min_days_greedy(n)}, tối ưu = {sol.minDays(n)}, lịch tối ưu = {min_days_with_schedule(n)[1]}")

Các n mà tham lam sai (n < 200): [12, 20, 25, 26, 43, 44, 60, 64, 83, 87] ...
n = 12: tham lam = 8, tối ưu = 7, lịch tối ưu = [3, 3]


Phản ví dụ nhỏ nhất chính là **ví dụ 3 của đề**, `n = 12`:

- Tham lam lấy `10` (chuỗi 4), còn `2 = 1 + 1` → chuỗi `4, 1, 1` → `4 + 1 + 1 + 2 nghỉ = 8` ngày.
- Tối ưu: `12 = 6 + 6` (hai chuỗi 3) → `3 + 3 + 1 nghỉ = 7` ngày.

Với `n = 20` còn lệch nhiều hơn: tham lam `15 + 3 + 1 + 1` → `5 + 2 + 1 + 1 + 3 nghỉ = 12` ngày, trong khi `20 = 10 + 10` chỉ cần `4 + 4 + 1 = 9` ngày.

Lấy "miếng lớn nhất" ngay lúc đầu có thể để lại phần dư rất khó ghép (phải dùng nhiều chuỗi ngắn, tốn nhiều ngày nghỉ).

## "Mọi số là tổng của tối đa 3 số tam giác" (Gauss, 1796)

Định lý đúng! Nhưng **tối thiểu số chuỗi ≠ tối thiểu số ngày**. Có khi dùng **4 chuỗi** lại ít ngày hơn mọi cách dùng `<= 3` chuỗi.

In [14]:
def best_with_at_most_3_streaks(n):
    tri = {triangular(L): L for L in range(1, 500) if triangular(L) <= n}
    best = tri.get(n, float("inf"))                      # 1 chuỗi
    for t1, a in tri.items():
        if n - t1 in tri:                                # 2 chuỗi
            best = min(best, a + tri[n - t1] + 1)
        for t2, b in tri.items():
            if n - t1 - t2 in tri:                       # 3 chuỗi
                best = min(best, a + b + tri[n - t1 - t2] + 2)
    return best

for n in [53, 86, 125]:
    print(f"n = {n}: tốt nhất với <= 3 chuỗi = {best_with_at_most_3_streaks(n)}, "
          f"tối ưu thật = {sol.minDays(n)}, lịch tối ưu = {min_days_with_schedule(n)[1]}")

n = 53: tốt nhất với <= 3 chuỗi = 18, tối ưu thật = 17, lịch tối ưu = [1, 1, 3, 9]
n = 86: tốt nhất với <= 3 chuỗi = 21, tối ưu thật = 20, lịch tối ưu = [1, 1, 3, 12]
n = 125: tốt nhất với <= 3 chuỗi = 24, tối ưu thật = 22, lịch tối ưu = [1, 1, 2, 15]


`dict.get(key, default)` trả về giá trị của `key` nếu có, ngược lại trả về `default` — tránh `KeyError`.

Bài học: đừng "đoán" cấu trúc lời giải tối ưu nếu chưa chứng minh. DP xét **mọi** khả năng nên luôn đúng.

# 11. Vì sao DP đúng?

1. **Mô hình đúng:** lịch tối ưu không có ngày nghỉ thừa (mục 3), nên mỗi lịch tối ưu ↔ một bộ chuỗi `(L1, ..., Lk)` với `Σ T(Li) = n`, số ngày `Σ (Li + 1) - 1`. Ngược lại, mọi bộ chuỗi như vậy đều dựng được lịch hợp lệ.
2. **Cấu trúc con tối ưu:** nếu bộ chuỗi tối ưu cho `s` có một chuỗi dài `L`, thì các chuỗi còn lại phải là bộ tối ưu cho `s - T(L)` (nếu không, thay bằng bộ tốt hơn sẽ được kết quả tốt hơn cho `s` — mâu thuẫn).
3. **Xét đủ:** công thức `dp[s] = min_L dp[s - T(L)] + L + 1` thử mọi khả năng cho "một chuỗi nào đó" trong bộ.
4. **Thứ tự tính:** `s - T(L) < s`, nên khi tính `dp[s]` mọi giá trị cần dùng đã có.

# 12. Độ phức tạp

Gọi `m ≈ √(2n)` là số đồng xu (số `L` có `T(L) <= n`).

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| 1. Đệ quy vét cạn | hàm mũ | độ sâu đệ quy `O(n)` |
| 2. Memoization | `O(n·m)` | `O(n)` + stack đệ quy sâu |
| 3. DP bottom-up | `O(n·m) = O(n√n)` | `O(n)` |
| 4. Tính trước một lần | `O(N√N)` một lần (`N = 10^5`), mỗi truy vấn `O(1)` | `O(N)` |

Với `n = 10^5`: `n·m ≈ 4.5·10^7` phép tính trong trường hợp xấu nhất; thực tế vòng lặp trong `break` sớm với `s` nhỏ nên chỉ khoảng `3·10^7`.

# 13. Những lỗi thường gặp

## Lỗi 1 — Quên trừ 1 ngày nghỉ cuối

Trả về `dp[n]` thay vì `dp[n] - 1` → mọi đáp án thừa 1.

## Lỗi 2 — Quên ngày nghỉ hoàn toàn

Dùng giá `L` thay vì `L + 1` → coi như các chuỗi nối liền nhau không cần nghỉ. Với `n = 2` ra `2` (sai, đúng là 3).

## Lỗi 3 — Dùng tham lam

Như mục 10, sai với `n = 20`.

## Lỗi 4 — Đệ quy sâu không kiểm soát

`RecursionError` với `n` lớn.

## Lỗi 5 — Dùng `/` thay vì `//`

`L * (L + 1) / 2` trả về số thực (`float`), dùng làm chỉ số list sẽ lỗi `TypeError`.

In [15]:
def wrong_no_rest(n):
    dp = [0] + [float("inf")] * n
    for s in range(1, n + 1):
        L = 1
        while triangular(L) <= s:
            dp[s] = min(dp[s], dp[s - triangular(L)] + L)     # quên ngày nghỉ
            L += 1
    return dp[n]

print("n = 2 | Sai (quên nghỉ):", wrong_no_rest(2), "| Đúng:", sol.minDays(2))
print("3 * 4 / 2 =", 3 * 4 / 2, type(3 * 4 / 2).__name__, "| 3 * 4 // 2 =", 3 * 4 // 2)

n = 2 | Sai (quên nghỉ): 2 | Đúng: 3
3 * 4 / 2 = 6.0 float | 3 * 4 // 2 = 6


# 14. Mẹo nhận diện pattern

- "Ghép các **mảnh** (được lặp lại) để có **đúng** tổng `n`, tối thiểu **chi phí**" → **Unbounded Knapsack / Coin Change**: `dp[s] = min(dp[s - giá_trị] + chi_phí)`.
- Các mảnh có giá trị tăng nhanh (`L²`, `2^L`, ...) → số mảnh ít (`√n`, `log n`) → DP `O(n·số_mảnh)` chạy được.
- Có **chi phí phụ giữa các mảnh** (ngày nghỉ) → gộp vào mỗi mảnh rồi bù ở cuối.
- Thấy đệ quy có **bài toán con lặp lại** → memoization hoặc bottom-up.
- Tham lam "lấy mảnh lớn nhất" thường **sai** với hệ mảnh không đặc biệt — luôn thử tìm phản ví dụ bằng máy.

# 15. Biến thể — Mỗi lần reset phải nghỉ `r` ngày

Nếu nghỉ một lần phải mất `r` ngày liền (ví dụ `r = 2`), chỉ cần đổi giá mỗi chuỗi thành `L + r`, và cuối cùng trừ `r`.

In [16]:
def min_days_rest_r(n, r):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        L = 1
        while triangular(L) <= s:
            dp[s] = min(dp[s], dp[s - triangular(L)] + L + r)
            L += 1
    return dp[n] - r

print("r = 1 (đề gốc):", [min_days_rest_r(n, 1) for n in (2, 9, 12)])
print("r = 2         :", [min_days_rest_r(n, 2) for n in (2, 9, 12)])

r = 1 (đề gốc): [3, 6, 7]
r = 2         : [4, 7, 8]


# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính số ngày ít nhất cho `n = 1, 2, ..., 8` bằng cách lập bảng `dp` như mục 7.

## Bài 2 — Dự đoán output

```python
[triangular(L) for L in range(1, 9)]
sum(1 for L in range(1, 100) if triangular(L) <= 50)
```

In [17]:
print([triangular(L) for L in range(1, 9)])
print(sum(1 for L in range(1, 100) if triangular(L) <= 50))

[1, 3, 6, 10, 15, 21, 28, 36]
9


## Bài 3 — Điền chỗ trống: DP bottom-up

In [18]:
def exercise_3_min_days(n):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    # TODO 1: duyệt s = 1..n
    # TODO 2: duyệt L = 1, 2, ... với triangular(L) <= s
    # TODO 3: dp[s] = min(dp[s], dp[s - triangular(L)] + L + 1)
    # TODO 4: return dp[n] - 1
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert [exercise_3_min_days(n) for n in (2, 9, 12)] == [3, 6, 7]

## Bài 4 — Truy vết

Viết hàm trả về **danh sách độ dài các chuỗi** trong một lịch tối ưu (thứ tự bất kỳ). Kiểm tra: tổng `T(L)` bằng `n` và `Σ(L + 1) - 1` bằng `minDays(n)`.

In [19]:
def exercise_4_streaks(n):
    # TODO: lưu choice[s], rồi đi ngược từ s = n
    raise NotImplementedError

## Bài 5 — Biến thể: điểm là số chính phương

Giả sử ngày thứ `k` của chuỗi được `2k - 1` điểm (`1, 3, 5, ...`), nên chuỗi dài `L` được `L²` điểm. Ngày nghỉ vẫn như cũ. Tìm số ngày ít nhất để có đúng `n` điểm.

In [20]:
def exercise_5_square_points(n):
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: đếm số lịch tối ưu

Đếm xem có **bao nhiêu bộ chuỗi khác nhau** (không tính thứ tự, ví dụ `[3, 2]` và `[2, 3]` là một) đạt số ngày tối thiểu.

Gợi ý: dùng DP kiểu "Coin Change II" — duyệt **đồng xu ở vòng ngoài** để không đếm trùng hoán vị, và lưu cặp `(chi phí nhỏ nhất, số cách)`.

In [21]:
def exercise_6_count_optimal(n):
    # TODO
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 3**
- Mức 1: giống hệt `min_days_dp`.
- Mức 2: `while triangular(L) <= s:` rồi `L += 1`.
- Mức 3: đừng quên `- 1` ở cuối.

**Bài 4**
- Mức 1: khi cập nhật `dp[s]` thành giá trị nhỏ hơn, ghi `choice[s] = L`.
- Mức 2: `while s > 0: streaks.append(choice[s]); s -= triangular(choice[s])`.

**Bài 5**
- Mức 1: chỉ đổi "đồng xu" từ `T(L)` sang `L*L`; giá vẫn `L + 1`.
- Mức 2: `L*L <= s`.

**Bài 6**
- Mức 1: `best[s]` = chi phí nhỏ nhất, `ways[s]` = số bộ đạt chi phí đó.
- Mức 2: vòng ngoài duyệt `L`, vòng trong duyệt `s` từ `T(L)` tới `n` (tăng dần — cho phép dùng lại `L`).
- Mức 3: nếu `best[s - t] + L + 1 < best[s]`: gán mới, `ways[s] = ways[s - t]`; nếu bằng: `ways[s] += ways[s - t]`.

# 17. Đáp án tham khảo

**Bài 1:** `dp[1..8] = 2, 4, 3, 5, 7, 4, 6, 8` → số ngày `= 1, 3, 2, 4, 6, 3, 5, 7`.

| n | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| ngày | 1 | 3 | 2 | 4 | 6 | 3 | 5 | 7 |
| lịch | `1` | `1+1` | `3` | `3+1` | `3+1+1` | `6` | `6+1` | `6+1+1` |

**Bài 2:** `[1, 3, 6, 10, 15, 21, 28, 36]` và `9` (vì `T(9) = 45 <= 50 < 55 = T(10)`).

In [22]:
assert [sol.minDays(n) for n in range(1, 9)] == [1, 3, 2, 4, 6, 3, 5, 7]
print("Bảng đáp án bài 1 đã được kiểm chứng ✔")

Bảng đáp án bài 1 đã được kiểm chứng ✔


In [23]:
def answer_3_min_days(n):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        L = 1
        while triangular(L) <= s:
            dp[s] = min(dp[s], dp[s - triangular(L)] + L + 1)
            L += 1
    return dp[n] - 1

assert [answer_3_min_days(n) for n in (2, 9, 12)] == [3, 6, 7]
assert all(answer_3_min_days(n) == sol.minDays(n) for n in range(1, 300))
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [24]:
def answer_4_streaks(n):
    INF = float("inf")
    dp = [INF] * (n + 1)
    choice = [0] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        L = 1
        while triangular(L) <= s:
            v = dp[s - triangular(L)] + L + 1
            if v < dp[s]:
                dp[s], choice[s] = v, L
            L += 1
    streaks, s = [], n
    while s > 0:
        streaks.append(choice[s])
        s -= triangular(choice[s])
    return streaks

for n in [2, 9, 12, 53, 777]:
    st = answer_4_streaks(n)
    assert sum(triangular(L) for L in st) == n
    assert sum(L + 1 for L in st) - 1 == sol.minDays(n)
    print(f"n = {n}: chuỗi {st}")
print("Đáp án bài 4 PASS")

n = 2: chuỗi [1, 1]
n = 9: chuỗi [2, 3]
n = 12: chuỗi [3, 3]
n = 53: chuỗi [1, 1, 3, 9]
n = 777: chuỗi [8, 38]
Đáp án bài 4 PASS


In [25]:
def answer_5_square_points(n):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for s in range(1, n + 1):
        L = 1
        while L * L <= s:
            dp[s] = min(dp[s], dp[s - L * L] + L + 1)
            L += 1
    return dp[n] - 1

def square_points_bruteforce(n):
    @lru_cache(maxsize=None)
    def g(rem):
        if rem == 0:
            return 0
        return min((L + 1) + g(rem - L * L) for L in range(1, int(rem ** 0.5) + 1))
    return g(n) - 1

for n in range(1, 200):
    assert answer_5_square_points(n) == square_points_bruteforce(n)
print("Đáp án bài 5 PASS:", [answer_5_square_points(n) for n in range(1, 11)])

Đáp án bài 5 PASS: [1, 3, 5, 2, 4, 6, 8, 5, 3, 5]


In [26]:
def answer_6_count_optimal(n):
    INF = float("inf")
    best = [INF] * (n + 1)
    ways = [0] * (n + 1)
    best[0], ways[0] = 0, 1
    L = 1
    while triangular(L) <= n:                  # vòng ngoài: đồng xu
        t, c = triangular(L), L + 1
        for s in range(t, n + 1):              # vòng trong: tổng, tăng dần
            v = best[s - t] + c
            if v < best[s]:
                best[s], ways[s] = v, ways[s - t]
            elif v == best[s]:
                ways[s] += ways[s - t]
        L += 1
    return ways[n]

def count_optimal_bruteforce(n):
    target = sol.minDays(n) + 1
    tri = [L for L in range(1, n + 1) if triangular(L) <= n]
    def go(rem, max_idx, cost):                # chọn chuỗi không tăng dần để tránh trùng
        if cost > target:
            return 0
        if rem == 0:
            return 1 if cost == target else 0
        total = 0
        for idx in range(max_idx, -1, -1):
            L = tri[idx]
            if triangular(L) <= rem:
                total += go(rem - triangular(L), idx, cost + L + 1)
        return total
    return go(n, len(tri) - 1, 0)

for n in range(1, 60):
    assert answer_6_count_optimal(n) == count_optimal_bruteforce(n), n
print("Đáp án bài 6 PASS")
print("Một vài n có nhiều hơn 1 lịch tối ưu:", [(n, answer_6_count_optimal(n)) for n in range(1, 200) if answer_6_count_optimal(n) > 1][:6])

Đáp án bài 6 PASS
Một vài n có nhiều hơn 1 lịch tối ưu: [(30, 2), (33, 3), (38, 2), (41, 2), (49, 2), (50, 2)]


# 18. Tổng kết

## Kiến thức Python

```python
L * (L + 1) // 2                  # số tam giác, dùng //
@lru_cache(maxsize=None)          # memoization
sys.getrecursionlimit()           # giới hạn đệ quy
try / except RecursionError
1 << 60                           # "vô cùng" kiểu số nguyên
dict.get(key, default), ", ".join(...)
```

## Kỹ thuật thuật toán

1. **Mô hình hoá**: lịch ↔ bộ số tam giác có tổng `n`.
2. **Gộp chi phí phụ** (ngày nghỉ) vào mỗi mảnh, bù ở cuối (`- 1`).
3. Đệ quy → **memoization** → **bottom-up DP** (Coin Change).
4. **Truy vết** bằng mảng `choice`.
5. **Tính trước một lần** cho mọi truy vấn.

## Công thức nhớ nhanh

```text
dp[0] = 0
dp[s] = min over L (T(L) <= s) of dp[s - T(L)] + L + 1
answer = dp[n] - 1
```

## Độ phức tạp

```text
Time:  O(n √n)   (446 đồng xu khi n = 10^5)
Space: O(n)
```

## Checklist trước khi nộp

- [ ] Giá mỗi chuỗi là `L + 1` (đã gộp ngày nghỉ)?
- [ ] Trả về `dp[n] - 1`?
- [ ] Dùng `//` cho số tam giác?
- [ ] Không dùng tham lam / không giả định "tối đa 3 chuỗi"?
- [ ] Tính bảng một lần nếu có nhiều test?

# 19. Phiên bản cuối cùng nên nhớ

```python
def _build(limit=10**5):
    points, costs = [], []
    L = 1
    while L * (L + 1) // 2 <= limit:
        points.append(L * (L + 1) // 2)
        costs.append(L + 1)
        L += 1
    dp = [0] * (limit + 1)
    for s in range(1, limit + 1):
        best = 1 << 60
        for p, c in zip(points, costs):
            if p > s:
                break
            if dp[s - p] + c < best:
                best = dp[s - p] + c
        dp[s] = best
    return dp

_DP = _build()

class Solution:
    def minDays(self, n: int) -> int:
        return _DP[n] - 1
```

Mấu chốt để nhớ: **mỗi chuỗi = một "đồng xu" mệnh giá `T(L)`, giá `L + 1`** — phần còn lại là bài Coin Change quen thuộc.